In [10]:
import numpy as np

def sigmoid(x): # 시그모이드 함수
    return 1. / (1. + np.exp(-x))

In [11]:
def numerical_derivative(f, x):
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [21]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata):   # 생성자
        self.name = gate_name
        self.xdata = xdata.reshape(4, 2)  # 입력층 데이터
        self.tdata = tdata.reshape(4, 1)  # 정답 데이터


        # 입력층(2개) -> 첫번째 은닉층(4개)
        self.W2 = np.random.randn(2, 4)
        self.b2 = np.zeros(4)

        # 첫번째 은닉층(4개) -> 두번째 은닉층(2개)
        self.W3 = np.random.randn(4, 2)
        self.b3 = np.zeros(2)

        # 두번째 은닉층(2개) -> 출력층(1개)
        self.W4 = np.random.randn(2, 1)
        self.b4 = np.zeros(1)
        
        self.learning_rate = 1e-1  # 학습율

    def loss_val(self):
        delta = 1e-7

        # 입력층 -> 첫번째 은닉층
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)

        # 첫번째 은닉층 -> 두번째 은닉층
        z3 = np.dot(a2, self.W3) + self.b3
        a3 = sigmoid(z3)

        # 두번째 은닉층 -> 출력층
        z4 = np.dot(a3, self.W4) + self.b4
        y = sigmoid(z4)

        return -np.sum(self.tdata*np.log(y+delta)+(1-self.tdata)*np.log((1-y)+delta))

    def train(self):
        f = lambda x : self.loss_val()
        print('Initial loss value =', self.loss_val())

        for step in range(11001):
            self.W2 -= self.learning_rate * numerical_derivative(f, self.W2)
            self.b2 -= self.learning_rate * numerical_derivative(f, self.b2)
            self.W3 -= self.learning_rate * numerical_derivative(f, self.W3)
            self.b3 -= self.learning_rate * numerical_derivative(f, self.b3)
            self.W4 -= self.learning_rate * numerical_derivative(f, self.W4)
            self.b4 -= self.learning_rate * numerical_derivative(f, self.b4)

            if (step % 1000 == 0):
                print('step =', step, ', loss value =', self.loss_val())


    def predict(self, input_data):
        self.xdata = input_data

        # 입력층 -> 첫번째 은닉층
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)

        # 첫번째 은닉층 -> 두번째 은닉층
        z3 = np.dot(a2, self.W3) + self.b3
        a3 = sigmoid(z3)

        # 두번째 은닉층 -> 출력층
        z4 = np.dot(a3, self.W4) + self.b4
        y = sigmoid(z4)

        if  y > 0.5:
            result = 1
        else:
            result = 0
        return y, result

    def accuracy(self, test_xdata, test_tdata):
        matched_list = []  # 정답과 일치한 내용 저장
        not_matched_list = []  # 정답과 불일치한 내용 저장

        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index)  # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index)  # 정답과 불일치한 내용 추가

        accuracy_val = len(matched_list) / len(test_xdata)  # 정확도 계산

        return accuracy_val

In [22]:
# XOR 게이트
# 첫번째 은닉층 노드 4개, 두번째 은닉층 노드 2개
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 0])   # 정답(target) 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)   # 모델 객체 생성
XOR_obj.train()   # 학습

Initial loss value = 2.7418928261267257
step = 0 , loss value = 2.7397412030421693
step = 1000 , loss value = 0.08560947047463274
step = 2000 , loss value = 0.019026028369821965
step = 3000 , loss value = 0.010248863900016017
step = 4000 , loss value = 0.00694105297867959
step = 5000 , loss value = 0.005224992192200442
step = 6000 , loss value = 0.004179935149888759
step = 7000 , loss value = 0.0034786410092560276
step = 8000 , loss value = 0.0029763087437202564
step = 9000 , loss value = 0.0025992148889399224
step = 10000 , loss value = 0.0023059490896998945
step = 11000 , loss value = 0.0020714920671653595


In [23]:
test_data = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 1, 1, 0])
accuracy_ret = XOR_obj.accuracy(test_data, test_tdata)
print('정확도 =>' , accuracy_ret)

[0 0] = 0
[0 1] = 1
[1 0] = 1
[1 1] = 0
----------------------------------------
정확도 => 1.0
